In [1]:
from architector import convert_io_molecule,view_structures
from architector.io_align_mol import reorder_align_rmsd
from architector.io_calc import CalcExecutor
from architector.io_molecule import convert_io_molecule
import architector.io_ptable as io_ptable

from ase.constraints import Hookean,ExternalForce
import numpy as np
import copy

def check_bonds(mol,
                bonds_breaking,
                bonds_forming,
                breaking_cutoff,
                forming_cutoff): # Convergence check function
    dists = mol.ase_atoms.get_all_distances()
    anums = mol.ase_atoms.get_atomic_numbers()
    goods = []
    for inds in bonds_breaking:
        cutoff_dist = (io_ptable.rcov1[anums[inds[0]]] + io_ptable.rcov1[anums[inds[1]]])*breaking_cutoff
        actual_dist = dists[inds[0]][inds[1]]
        if actual_dist > cutoff_dist:
            goods.append(True)
        else:
            goods.append(False)
    for inds in bonds_forming:
        cutoff_dist = (io_ptable.rcov1[anums[inds[0]]] + io_ptable.rcov1[anums[inds[1]]])*forming_cutoff
        actual_dist = dists[inds[0]][inds[1]]
        if actual_dist < cutoff_dist:
            goods.append(True)
        else:
            goods.append(False)
    return np.all(goods)


In [2]:
mol_init = convert_io_molecule('r3.xyz')
mol_final = convert_io_molecule('p3.xyz')
view_structures([mol_init,mol_final])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
mol_init.detect_charge_spin()
print(mol_init.uhf,mol_init.charge)
mol_final.detect_charge_spin()
print(mol_final.uhf,mol_final.charge)

0 0
0 0


In [4]:
mol1_relaxed = CalcExecutor(mol_init, relax=False).mol
view_structures(mol1_relaxed)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
mol2 = CalcExecutor(mol_final, relax=True, save_trajectories=True,maxsteps=20)
# view_structures(mol2_relaxed)

In [12]:
mol2 = CalcExecutor(mol_final, relax=True, save_trajectories=True,maxsteps=2)
view_structures(mol2.trajectory)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
mol2 = CalcExecutor(mol_final, relax=True, save_trajectories=True,maxsteps=10)
view_structures(mol2.trajectory)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
mol_final_ats = reorder_align_rmsd(mol_init.ase_atoms, mol_final.ase_atoms, sample=1000)
mol_final = convert_io_molecule(mol_final_ats)
view_structures([mol_init, mol_final], labelinds=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
breaking_cutoff=1.5 # When a bond is breaking, what the distance should be
forming_cutoff=1.2 # When a bond is forming, what the distance should be (Angstroms)
start_force_constant=0.05 # eV/angstrom
force_increment=0.05 # How fast to ramp up the force constant
structure_match_fconst=0.001 # Add springs to force the inial geometry towards the final geometry?
method='GFN2-xTB' # XTB
max_steps=4 # Steps/opimization iteration
fmax_opt=0.1 # Cutoff for the maximum force.
initial = mol_init # Initial configuration
final = mol_final # Final Configuration.
mol1 = convert_io_molecule(initial)
mol2 = convert_io_molecule(final)
mol1.create_mol_graph()
mol2.create_mol_graph()
# Find the formed bonds
bonds_forming = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == 1)) if x[0] < x[1]]
# Find the broken bonds
bonds_breaking = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == -1)) if x[0] < x[1]]
fconst = start_force_constant
save_trajectory = []
opt_mol = copy.deepcopy(mol1)
keep_going = True # Exit flag for loop
while keep_going:
    print('Running Fconst = {}'.format(fconst))
    opt_mol.ase_atoms.set_constraint()
    constraints = []
    for inds in bonds_forming: # Add hookean constants
        constraint = ExternalForce(inds[0], inds[1], -fconst)
        constraints.append(constraint)
    for inds in bonds_breaking:
        constraint = ExternalForce(inds[0], inds[1], fconst)
        constraints.append(constraint)
    for ind in range(mol1.graph.shape[0]): # Add distance setting 
        constraint = Hookean(ind, mol2.ase_atoms.positions[ind],
                                k=structure_match_fconst,
                                rt=0.1)
        constraints.append(constraint)
    opt_mol.ase_atoms.set_constraint(constraints)
    tmpopt = CalcExecutor(opt_mol,
                            method=method,
                            relax=True,
                            fmax=fmax_opt,
                            maxsteps=max_steps,
                            use_constraints=True)
    tmpopt.mol.ase_atoms.calc = None
    save_trajectory.append(copy.deepcopy(tmpopt.mol))
    opt_mol = tmpopt.mol
    good = check_bonds(opt_mol, bonds_breaking, bonds_forming,
                        breaking_cutoff, forming_cutoff)
    if good:
        keep_going = False
    else:
        fconst += force_increment

Running Fconst = 0.05
Running Fconst = 0.1
Running Fconst = 0.15000000000000002
Running Fconst = 0.2


In [11]:
view_structures(save_trajectory)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
from ase.io import read # Read in the initial and final molecules.
mols = read('R19.xyz',index=':')
mol_init = convert_io_molecule(mols[0])
mol_final = convert_io_molecule(mols[1])
view_structures([mol_init,mol_final])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
mol_init.detect_charge_spin()
print(mol_init.uhf,mol_init.charge)
mol_final.detect_charge_spin()
print(mol_final.uhf,mol_final.charge)
# Get the charge/spin parameters

0 0
0 0


In [16]:
breaking_cutoff=1.5 # When a bond is breaking, what the distance should be
forming_cutoff=1.2 # When a bond is forming, what the distance should be (Angstroms)
start_force_constant=0.05 # eV/angstrom
force_increment=0.1 # How fast to ramp up the force constant
structure_match_fconst=0.000 # Add springs to force the inial geometry towards the final geometry?
method='GFN2-xTB' # XTB
max_steps=4 # Steps/opimization iteration
fmax_opt=0.1 # Cutoff for the maximum force.
initial = mol_init # Initial configuration
final = mol_final # Final Configuration.
mol1 = convert_io_molecule(initial)
mol2 = convert_io_molecule(final)
mol1.create_mol_graph()
mol2.create_mol_graph()
# Find the formed bonds
bonds_forming = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == 1)) if x[0] < x[1]]
# Find the broken bonds
bonds_breaking = [(int(x[0]), int(x[1])) for x in zip(*np.where((mol2.graph - mol1.graph) == -1)) if x[0] < x[1]]
print(bonds_breaking)
print(bonds_forming)
fconst = start_force_constant
save_trajectory = []
opt_mol = copy.deepcopy(mol1)
keep_going = True # Exit flag for loop
nstep = 0 
while keep_going:
    print('Running Fconst = {}'.format(fconst))
    opt_mol.ase_atoms.set_constraint()
    nstep += 1
    constraints = []
    for inds in bonds_forming: # Add hookean constants
        constraint = ExternalForce(inds[0], inds[1], -fconst)
        constraints.append(constraint)
    for inds in bonds_breaking:
        constraint = ExternalForce(inds[0], inds[1], fconst/nstep)
        constraints.append(constraint)
    for ind in range(mol1.graph.shape[0]): # Add distance setting 
        constraint = Hookean(ind, mol2.ase_atoms.positions[ind],
                                k=structure_match_fconst,
                                rt=0.1)
        constraints.append(constraint)
    opt_mol.ase_atoms.set_constraint(constraints)
    tmpopt = CalcExecutor(opt_mol,
                            method=method,
                            relax=True,
                            fmax=fmax_opt,
                            maxsteps=max_steps,
                            use_constraints=True,
                            debug=True)
    tmpopt.mol.ase_atoms.calc = None
    save_trajectory.append(copy.deepcopy(tmpopt.mol))
    opt_mol = tmpopt.mol
    good = check_bonds(opt_mol, bonds_breaking, bonds_forming,
                        breaking_cutoff, forming_cutoff)
    if good:
        keep_going = False
    else:
        fconst += force_increment

[(6, 14), (7, 14), (11, 89), (14, 67)]
[(11, 14), (14, 89)]
Running Fconst = 0.05
No actinides present to swap.
------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -112.1478276066  -1.1364239E+02   1.3574313E-01
      2     -112.2488839206  -1.0105631E-01   7.7194250E-02
      3     -112.2244337099   2.4450211E-02   3.1143540E-02
      4     -112.2625884764  -3.8154766E-02   1.2634258E-02
      5     -112.2667979396  -4.2094633E-03   6.5913740E-03
      6     -112.2680693786  -1.2714390E-03   3.7015502E-03
      7     -112.2680902413  -2.0862653E-05   2.7180912E-03
      8     -112.2683075816  -2.1734036E-04   1.6970434E-03
      9     -112.2684148499  -1.0726823E-04   7.6777082E-04
     10     -112.2684274724  -1.2622553E-05   4.3820832E-04
     11     -112.2684279436  -4.7118282E-07   2.5372086E-04
     12     -112.2684292928  -1.3491823E-06   

Exception ignored from cffi callback <function logger_callback at 0x167bc30a0>:
Traceback (most recent call last):
  File "/Users/mgt16/mambaforge/lib/python3.10/site-packages/tblite/library.py", line 78, in logger_callback
    @ffi.def_extern()
KeyboardInterrupt: 


No actinides present to swap.
Running Fconst = 0.5499999999999999
No actinides present to swap.
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -112.1675981837  -1.1384147E+02   1.3769064E-01
      2     -112.2669681662  -9.9369982E-02   7.9450975E-02
      3     -112.2468967504   2.0071416E-02   2.9658554E-02
      4     -112.2793948391  -3.2498089E-02   1.2200558E-02
      5     -112.2824310543  -3.0362153E-03   6.9731544E-03
      6     -112.2834113241  -9.8026976E-04   4.2124576E-03
      7     -112.2835282182  -1.1689408E-04   2.8711385E-03
      8     -112.2837682565  -2.4003835E-04   1.6389784E-03
      9     -112.2838536017  -8.5345194E-05   7.7127846E-04
     10     -112.2838649470  -1.1345289E-05   4.3371592E-04
     11     -112.2838655952  -6.4823253E-07   2.2997222E-04
     12     -112.2838667011  -1.1059038E-06   1.0766452E-04
     13     -112.2838669212  -2.2004733E-07   7.0854136E-05
   

In [15]:
view_structures(save_trajectory,trajectory=True,interval=500)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.